In [ ]:
from google.colab import drive
import os
import zipfile

drive.mount('/content/drive')

zip_path = '/content/drive/MyDrive/thermal_camera_baseline.zip'
extract_path = '/content/baseline_dataset_working_dir'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Zip başarıyla çıkarıldı!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zip başarıyla çıkarıldı!


In [ ]:
!pip install ultralytics

In [ ]:
import os
import yaml
from ultralytics import YOLO

# 1. data.yaml yolu
yaml_path = '/content/baseline_dataset_working_dir/thermal_camera_baseline/dataset/data.yaml'

# 2. YAML dosyasını oku ve path kısmını Colab yoluna göre güncelle
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

# 'path' anahtarını Colab'deki dataset klasörünün tam yoluyla değiştiriyoruz
dataset_folder_path = os.path.dirname(yaml_path)
data_config['path'] = dataset_folder_path

# Güncellenmiş veriyi tekrar data.yaml içine yaz
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f"✅ data.yaml başarıyla güncellendi! Yeni path: {dataset_folder_path}")

model = YOLO('yolov8n.pt')

results = model.train(
    data = yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    name='thermal_human_v1'
)

✅ data.yaml başarıyla güncellendi! Yeni path: /content/baseline_dataset_working_dir/thermal_camera_baseline/dataset
Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/baseline_dataset_working_dir/thermal_camera_baseline/dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=

In [ ]:
'''
import os
import glob
import shutil

# 1. En son eğitilen modelin best.pt dosyasını bul
weight_files = glob.glob('/content/runs/detect/*/weights/best.pt')

if weight_files:
    # En son oluşturulan best.pt dosyasını seç
    latest_best = max(weight_files, key=os.path.getmtime)
    print(f" Bulunan en güncel ağırlık dosyası: {latest_best}")

    # Drive'a kopyala
    dst_best = '/content/drive/MyDrive/thermal_human_v1_best.pt'
    shutil.copy(latest_best, dst_best)
    print(" Model ağırlığı Drive'a başarıyla kopyalandı: thermal_human_v1_best.pt")
else:
    print(" best.pt dosyası bulunamadı. Eğitimin başarıyla tamamlandığından emin ol.")
'''

SyntaxError: incomplete input (2385958467.py, line 1)

In [ ]:
import os
import shutil
from pathlib import Path

# 1. runs/detect altındaki en güncel veya belirtilen klasörü tespit et
target_dir = None
possible_names = ['thermal_human_v1-4', 'thermal_human_v1_4']

# Önce doğrudan ismi kontrol et, yoksa runs/detect altındaki en son klasörü bul
for name in possible_names:
    path = f'/content/runs/detect/{name}'
    if os.path.exists(path):
        target_dir = path
        break

if not target_dir:
    # İsmi tam eşleşmezse en son oluşturulan klasörü otomatik bulur
    all_runs = sorted(Path('/content/runs/detect').glob('*'), key=os.path.getmtime)
    if all_runs:
        target_dir = str(all_runs[-1])

if target_dir:
    folder_name = os.path.basename(target_dir)
    dst_dir = f'/content/drive/MyDrive/{folder_name}'

    # Drive'da aynı isimde klasör varsa çakışmayı önlemek için silip günceller
    if os.path.exists(dst_dir):
        shutil.rmtree(dst_dir)

    shutil.copytree(target_dir, dst_dir)
    print(f"Proje klasörü komple Drive'a aktarıldı: MyDrive/{folder_name}")
else:
    print("Eğitime ait klasör bulunamadı. Lütfen runs/detect dizinini kontrol et.")

✅ Proje klasörü komple Drive'a aktarıldı: MyDrive/thermal_human_v1-4


In [ ]:
'''
import os

# Tüm dizinlerde data.yaml dosyasını ara
for root, dirs, files in os.walk('/content'):
    if 'data.yaml' in files:
        print("🎯 BULUNDU! Tam dosya yolu:")
        print(os.path.join(root, 'data.yaml'))
'''